# Week 2 — SST-2 (sentiment) + RoBERTa (N=1000)
**Project**: Quantum Kernels on CLIP/BERT Embeddings — Encoder Comparison

**This is the ROBERTA version of the SST2 embedding pipeline.**
Same structure as your original BERT Week 2 — only the encoder changed.

**Config:**
- Encoder: roberta-base (768-dim)
- Classes: 2
- Metric: binary F1
- Output tag: `sst2_roberta` (does not overwrite your BERT files)

**Runtime:** ~8 min on Colab T4

In [ ]:
!pip install datasets transformers torch sentence-transformers scikit-learn numpy --quiet

import numpy as np
import time, os, pickle, shutil, json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

---
## Step 0: Storage + checkpoint helpers

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
TAG = 'sst2_roberta'
print(f"Save directory: {SAVE_DIR}")
print(f"File tag: {TAG}")

def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")

def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)

def save_figure(fig, name):
    for ext in ('png','pdf'):
        p = f"{name}.{ext}"
        fig.savefig(p, bbox_inches='tight', dpi=150)
        _persist_and_download(p)

print("Helpers ready.")

---
## Step 1: Load SST2

In [ ]:
from datasets import load_dataset

LABELS = {0:'negative', 1:'positive'}
print("Loading sst2...")
dataset = load_dataset('nyu-mll/glue', 'sst2')

N_TRAIN_EXTRACT = 1100
N_TEST_EXTRACT  = 250

train_data = dataset['train'].shuffle(seed=42).select(range(N_TRAIN_EXTRACT))
test_data  = dataset['validation'].shuffle(seed=42).select(range(N_TEST_EXTRACT))

train_texts  = [x['sentence'] for x in train_data]
train_labels = np.array([x['label'] for x in train_data])
test_texts   = [x['sentence'] for x in test_data]
test_labels  = np.array([x['label'] for x in test_data])

print(f"Train {len(train_texts)} | Test {len(test_texts)}")
print(f"Classes: {sorted(set(train_labels))}")
for i in range(2):
    print(f"  [{LABELS[train_labels[i]]}] {train_texts[i][:65]}...")

---
## Step 2: Extract ROBERTA embeddings — checkpoint after

In [ ]:
# RoBERTa — first-token (CLS-style) pooling, same as BERT
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained('roberta-base')
model     = AutoModel.from_pretrained('roberta-base').to(DEVICE)
model.eval()
print(f"roberta-base loaded")

def extract(texts, batch_size=32):
    out = []
    nb = (len(texts) + batch_size - 1) // batch_size
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=128, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            o = model(**inputs)
        out.append(o.last_hidden_state[:, 0, :].cpu().numpy())
        if (i//batch_size+1) % 5 == 0: print(f"  batch {i//batch_size+1}/{nb}")
    return np.vstack(out)

print("Extracting train...")
t0 = time.time()
train_emb = extract(train_texts)
print(f"  {train_emb.shape} in {time.time()-t0:.0f}s")
print("Extracting test...")
test_emb = extract(test_texts)
print(f"  {test_emb.shape}")

assert train_emb.shape[1] == 768, "Unexpected embedding dim"
print(f"\nEmbedding dim confirmed: 768")

print("\nCheckpoint 1 — raw embeddings:")
checkpoint(f'{TAG}_raw.npz',
           lambda p: np.savez(p, train_emb=train_emb, test_emb=test_emb,
                              train_labels=train_labels, test_labels=test_labels))

---
## Step 3: PCA + normalize — checkpoint after

In [ ]:
N_QUBITS = 8
pca_dims = [4, 8, 12, 16]
pca_objects = {n: PCA(n_components=n, random_state=42).fit(train_emb) for n in pca_dims}

print("PCA variance explained (768-dim -> reduced):")
for n in pca_dims:
    print(f"  {n:>2} dims -> {np.sum(pca_objects[n].explained_variance_ratio_)*100:.1f}%")

pca_8 = pca_objects[N_QUBITS]
X_train_pca = pca_8.transform(train_emb)
X_test_pca  = pca_8.transform(test_emb)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_scaled = scaler.fit_transform(X_train_pca)
X_test_scaled  = np.clip(scaler.transform(X_test_pca), 0, np.pi)

print(f"\nTrain range: [{X_train_scaled.min():.3f}, {X_train_scaled.max():.3f}]")

print("\nCheckpoint 2 — PCA + scaler:")
checkpoint(f'{TAG}_pca_scaler.pkl',
           lambda p: pickle.dump({'pca': pca_8, 'scaler': scaler, 'n_qubits': N_QUBITS,
                                  'dataset': 'sst2', 'encoder': 'roberta',
                                  'n_classes': 2}, open(p,'wb')))

---
## Step 4: Classical baselines (metric = binary F1)

In [ ]:
N_TRAIN_SVM, N_TEST_SVM = 1000, 200
assert len(X_train_scaled) >= N_TRAIN_SVM
assert len(X_test_scaled)  >= N_TEST_SVM

idx_tr = np.random.RandomState(42).choice(len(X_train_scaled), N_TRAIN_SVM, replace=False)
idx_te = np.random.RandomState(42).choice(len(X_test_scaled),  N_TEST_SVM,  replace=False)
X_tr, y_tr = X_train_scaled[idx_tr], train_labels[idx_tr]
X_te, y_te = X_test_scaled[idx_te],  test_labels[idx_te]

def score_f1(yt, yp): return float(f1_score(yt, yp, average='binary'))

print(f"Train {X_tr.shape} | Test {X_te.shape} | metric=binary F1")
print()

classifiers = {
    'Linear':     SVC(kernel='linear', C=1.0, random_state=42),
    'RBF':        SVC(kernel='rbf',    C=1.0, gamma='scale', random_state=42),
    'Polynomial': SVC(kernel='poly',   C=1.0, degree=3, coef0=1, random_state=42),
}
results = {}
print(f"{'Kernel':<14} {'Accuracy':>10} {'F1':>10}")
print("-"*36)
for name, clf in classifiers.items():
    clf.fit(X_tr, y_tr)
    yp = clf.predict(X_te)
    results[name] = {'accuracy': float(accuracy_score(y_te, yp)), 'f1': score_f1(y_te, yp)}
    print(f"{name:<14} {results[name]['accuracy']:>10.4f} {results[name]['f1']:>10.4f}")

# bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
names = list(results.keys())
for ax, key, lbl in [(axes[0],'accuracy','Accuracy'), (axes[1],'f1','Binary F1')]:
    vals = [results[n][key] for n in names]
    ax.bar(names, vals, color=['steelblue','coral','mediumseagreen'], edgecolor='white')
    ax.set_ylim(0.4, 1.0); ax.set_title(lbl); ax.set_ylabel(lbl)
    for i, v in enumerate(vals): ax.text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=11)
plt.suptitle('SST-2 (sentiment) + RoBERTa — classical baselines (N=1000)')
plt.tight_layout(); plt.show()
save_figure(fig, f'{TAG}_baselines')

print("\nCheckpoint 3 — baselines:")
checkpoint(f'{TAG}_baseline_results.pkl', lambda p: pickle.dump(results, open(p,'wb')))

---
## Step 5: PCA dimension sweep

In [ ]:
print(f"  {'Dims':>4}  {'Var%':>6}  {'Acc':>8}  {'F1':>8}")
print("-"*34)
for n_dim in pca_dims:
    X_tr_d = pca_objects[n_dim].transform(train_emb[idx_tr])
    X_te_d = pca_objects[n_dim].transform(test_emb[idx_te])
    sc_d   = MinMaxScaler(feature_range=(0, np.pi))
    X_tr_d = sc_d.fit_transform(X_tr_d)
    X_te_d = np.clip(sc_d.transform(X_te_d), 0, np.pi)
    clf_d  = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42).fit(X_tr_d, y_tr)
    yp = clf_d.predict(X_te_d)
    var = np.sum(pca_objects[n_dim].explained_variance_ratio_)*100
    print(f"  {n_dim:>4}  {var:>5.1f}%  {accuracy_score(y_te,yp):>8.4f}  "
          f"{score_f1(y_te,yp):>8.4f}")

---
## Step 6: Save final embeddings — shape verified

In [ ]:
def save_final(p):
    np.savez(p, X_train=X_train_scaled, X_test=X_test_scaled,
             y_train=train_labels, y_test=test_labels,
             X_tr_subset=X_tr, X_te_subset=X_te,
             y_tr_subset=y_tr, y_te_subset=y_te)

print("Checkpoint 4 — final embeddings:")
checkpoint(f'{TAG}_embeddings.npz', save_final)

loaded = np.load(f'{TAG}_embeddings.npz')
print(f"\n  X_tr_subset = {loaded['X_tr_subset'].shape}  (must be (1000, 8))")
print(f"  X_te_subset = {loaded['X_te_subset'].shape}  (must be (200, 8))")
assert loaded['X_tr_subset'].shape == (1000, 8)
assert loaded['X_te_subset'].shape == (200, 8)
print("\n  VERIFIED.")

meta = {'dataset':'sst2', 'encoder':'roberta',
        'encoder_model':'roberta-base', 'embedding_dim':768,
        'n_classes':2, 'metric':'binary',
        'n_train':1000, 'n_test':200, 'baselines':results}
checkpoint(f'{TAG}_meta.json', lambda p: json.dump(meta, open(p,'w'), indent=2))

print(f"\nFiles for {TAG}:")
for f in sorted(os.listdir(SAVE_DIR)):
    if f.startswith(TAG): print(f"  {f}")

---
## Week 2 done — SST-2 (sentiment) + RoBERTa

### Saved (tag = `sst2_roberta`):
- `sst2_roberta_raw.npz` — raw 768-dim embeddings
- `sst2_roberta_pca_scaler.pkl`
- `sst2_roberta_baseline_results.pkl`
- `sst2_roberta_embeddings.npz` — for Week 3
- `sst2_roberta_meta.json`
- `sst2_roberta_baselines.png/.pdf`

### Next:
Week 3 for this config loads `sst2_roberta_embeddings.npz` and uses binary F1.
Your BERT files are untouched — everything here is tagged `sst2_roberta`.